# Figure 4 | Hippocampal LR communication landscape

Publication-ready analysis notebook for the manuscript figure. Exploratory checks, scratch outputs, and analyses unrelated to the figure panels have been removed. Paths are repository-relative; set `NVU_PROJECT_ROOT` when running from another location.

Analysis scope:
- Stereosite ligand-receptor scoring for hippocampal digital NVUs.
- Region-level LR pair count and communication-strength summaries.

In [ ]:
# Repository-relative paths
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("NVU_PROJECT_ROOT", Path.cwd().parent)).resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
REFERENCE_DIR = PROJECT_ROOT / "references"
FIGURE_DIR = RESULTS_DIR / "figure4"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
from stereosite.plot.scii import ligrec
import anndata
import os
from stereosite.scii import intensities_count
import sys
import stereosite
import matplotlib.pyplot as plt

samples_list = [
    'D01574C4', 'D01574C6', 'D03556C4', 'D03556D4',
    'D03556D6', 'D03556E2', 'D03556E4', 'D03556E6'
]

In [ ]:
import pandas as pd
import numpy as np
import anndata
import os
from stereosite.scii import intensities_count


filepath = '../results/05_nvu_analysis_results/Hip/200/'
interactiondb_file = "../references/stereosite/CellChatDB_Integrated.csv"
resultpath = '../results/07_Stereosite/test'
distance_threshold = 100
desired_areas = ['FAS','CA1','CA2','CA3','CA4','SLRM','DG']

# Ensure that the results directory exists
os.makedirs(resultpath, exist_ok=True)

# Loop over samples
for profile in samples_list:
    print(f"\n{'='*60}")
    print(f"Processing sample: {profile}")
    print(f"{'='*60}")
    
    # Build the data file path
    data_file = os.path.join(filepath, profile, 'nvu_filtered.h5ad')
    
    # Check whether the file exists
    if not os.path.exists(data_file):
        print(f"Warning: File not found for {profile}: {data_file}")
        print(f"Skipping {profile}...")
        continue
    
    try:
        # Load the data
        print(f"Loading data from: {data_file}")
        adata = anndata.read(data_file)
        
        # Combine 'x' and 'y' into spatial coordinates
        spatial_coords = np.vstack((adata.obs['x'], adata.obs['y'])).T
        adata.obsm['spatial'] = spatial_coords
        
        # Loop over regions
        for area in desired_areas:
            print(f"\n  Processing area: {area}")
            
            # Filter data
            filtered_data = adata[adata.obs['area_m'].isin([area])]
            
            # Check whether the filtered data are empty
            if filtered_data.n_obs == 0:
                print(f"    Warning: No data for area {area} in sample {profile}, skipping...")
                continue
            
            print(f"    Cells in area {area}: {filtered_data.n_obs}")
            
            # Run intensities_count
            print(f"    Computing cell-cell interactions...")
            scii_dict = intensities_count(
                filtered_data, 
                interactiondb_file, 
                distance_threshold=distance_threshold, 
                anno='celltype', 
                jobs=16
            )
            
            # Retrieve results
            intensity_df = scii_dict['intensities']
            pvalues_df = scii_dict['pvalues']
            
            # Define the output path
            intensity_csv_path = os.path.join(resultpath, f"{profile}_{area}_intensity_df.csv")
            pvalues_csv_path = os.path.join(resultpath, f"{profile}_{area}_pvalues_df.csv")
            
            # Save results
            intensity_df.to_csv(intensity_csv_path, index=True)
            pvalues_df.to_csv(pvalues_csv_path, index=True)
            
            print(f"    Results saved:")
            print(f"      - {intensity_csv_path}")
            print(f"      - {pvalues_csv_path}")
        
        print(f"\nCompleted processing for sample: {profile}")
        
    except Exception as e:
        print(f"Error processing sample {profile}: {str(e)}")
        import traceback
        traceback.print_exc()
        print(f"Continuing with next sample...")
        continue

print(f"\n{'='*60}")
print("All samples processed!")
print(f"{'='*60}")